# LEGO shape CNN — training on Kaggle

1. *Add data* → search **Images of LEGO Bricks** (Joost Hazelzet).
2. Settings → Accelerator **GPU**, Internet **on**.
3. Upload our crops as a private Kaggle dataset (`data/real/train`, `data/real/test`) and add it too.
4. Run all. Download `models/shape_cnn.pt` and `results/` at the end.

In [ ]:
!git clone https://github.com/YOUR-TEAM/lego-recognition.git
%cd lego-recognition
!pip install -q timm wandb

In [ ]:
import os, glob
print(glob.glob('/kaggle/input/*'))
RENDERS = '/kaggle/input/images-of-lego-bricks'   # adjust to what is printed above
REAL = '/kaggle/input/lego-real-crops'            # our own crops (private dataset)
HAS_REAL = os.path.isdir(REAL)

In [ ]:
# optional: experiment tracking
# import wandb; wandb.login()   # key from wandb.ai/authorize
WANDB = ''  # set to '--wandb' after login

## Quick sanity run (2 min)

In [ ]:
!python scripts/train_cnn.py --renders "$RENDERS" --limit 50 --epochs 2 --out /tmp/quick.pt --tag quick

## Core experiment: A → B → C

In [ ]:
test = f'--real-test {REAL}/test' if HAS_REAL else ''
!python scripts/train_cnn.py --renders "$RENDERS" $test --aug basic --epochs 15 --out models/A.pt --tag A_renders_basic $WANDB

In [ ]:
!python scripts/train_cnn.py --renders "$RENDERS" $test --aug real --epochs 15 --out models/B.pt --tag B_renders_real $WANDB

In [ ]:
if HAS_REAL:
    !python scripts/train_cnn.py --renders "$RENDERS" --real-train {REAL}/train --real-test {REAL}/test --aug real --epochs 15 --ft-epochs 10 --out models/shape_cnn.pt --tag C_finetuned $WANDB

In [ ]:
import pandas as pd, json
rows=[json.loads(l) for l in open('results/experiments.jsonl')]
pd.json_normalize(rows)

In [ ]:
if HAS_REAL:
    !python scripts/evaluate.py models/shape_cnn.pt {REAL}/test --tag C_finetuned
    from IPython.display import Image, display
    display(Image('results/eval_C_finetuned/confusion.png'))

In [ ]:
!zip -qr /kaggle/working/results.zip results models
print('download results.zip from the Output tab')